# L08-03　msame 推理验证与常见错误

L08-01 得到了 OM，L08-02 学会了写转换命令，但"转换成功不等于结果正确"。本节对 msame 基础用法进行讲解，给出报错的排查方法。

**前置**：已完成 L08-01、L08-02，`l08_workspace/demo_model.om` 已生成。

**环境**：本节大部分内容需要昇腾环境与 NPU。

## 1　msame 

名字来自 Model Same——验证转换前后的模型行为是否一致。它与 ATC 构成"转换—验证"闭环。

| 工具 | 用途 |
| --- | --- |
| ATC | 模型转换 |
| **msame** | **单次/多次推理验证** |
| Profiling | 性能瓶颈分析 |
| benchmark | 标准模型性能基准 |

msame 做的是功能验证和基础的耗时测量，不是精度评测工具，也不是压测工具。要评测数据集精度或做压力测试，用对应的专门工具。

获取方式：Gitee 的 `ascend/tools` 仓库，`msame` 目录。编译前需先配置 CANN 环境，然后按仓库 README 执行编译脚本，产物通常在 `out/msame`。

## 2　参数与最小用法

第一个动作是核对参数。msame 不同版本的参数有差异。

In [1]:
! /opt/atomgit/tools/msame/out/msame --help


Usage:
generate offline model inference output file example:
./msame --model /home/HwHiAiUser/ljj/colorization.om --input /home/HwHiAiUser/ljj/colorization_input.bin     --output /home/HwHiAiUser/ljj/AMEXEC/out/output1 --outfmt TXT --loop 2

arguments explain:
  --model       Model file path
  --input	Input data path(only accept binary data file) 	    If there are several file, please seprate by ','
  --output	Output path(User needs to have permission to create directories)
  --outfmt	Output file format (TXT or BIN)
  --loop 	loop time(must in 1 to 100)
  --dump	Enable dump (true or false)
  --profiler	Enable profiler (true or false)
  --device      Designated the device ID(must in 0 to 255)
  --debug       Debug switch,print model information (true or false)
  --outputSize  Set model output size, such as --outputSize "10000,10000"
  --dymBatch    dynamic batch size param，such as --dymBatch 2
  --dymHW       dynamic image size param, such as --dymHW "300,500"
  --dymDims 	dynamic dims

静态 shape 场景四个参数就够：

| 参数 | 含义 |
| --- | --- |
| `--model` | OM 模型路径 |
| `--input` | 输入数据，单个 bin 文件或目录 |
| `--output` | 输出目录，msame 在其下建时间戳子目录 |
| `--outfmt` | 输出格式，`BIN` 或 `TXT` |

In [2]:
# %%bash
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

! /opt/atomgit/tools/msame/out/msame --model l08_workspace/demo_model.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/msame_out \
      --outfmt BIN

[INFO] acl init success
[INFO] open device 0 success
[INFO] create context success
[INFO] create stream success
[INFO] get run mode success
[INFO] load model l08_workspace/demo_model.om success
[INFO] create model description success
[INFO] get input dynamic gear count success
[INFO] create model output success
l08_workspace/msame_out/2026814_16_5_28_817453
[INFO] start to process file:l08_workspace/input.bin
[INFO] model execute success
Inference time: 0.223ms
[INFO] get max dynamic batch size success
[INFO] output data success
Inference average time: 0.223000 ms
[INFO] destroy model input success
[INFO] unload model success, model Id is 1
[INFO] pid: 9597 Execute sample success
[INFO] end to destroy stream
[INFO] end to destroy context
[INFO] end to reset device is 0
[INFO] end to finalize acl


`--input` 可以省略，这时 msame 用自动生成的数据推理。这种模式只适合测耗时，因为随机输入可能掩盖问题——比如某个分支从未被触发。验证精度必须用真实数据。省略 `--input` 时的具体行为用 `--help` 确认。

## 3　输入输出

### 3.1　bin 文件的组织

bin 是裸二进制，没有 shape 也没有 dtype。单输入时一个文件；多输入时 msame 按输入顺序匹配文件，所以文件名的排序必须与模型输入顺序一致。

给文件名加数字前缀是可靠的做法：

In [3]:
import numpy as np, os

os.makedirs("l08_workspace/multi_in", exist_ok=True)
np.random.seed(0)

inputs = {
    "00_input_ids": (np.random.randint(0, 1000, (1, 128)).astype(np.int64), np.int64),
    "01_attention_mask": (np.ones((1, 128), dtype=np.int64), np.int64),
    "02_token_type_ids": (np.zeros((1, 128), dtype=np.int64), np.int64),
}

for name, (arr, dt) in inputs.items():
    path = "l08_workspace/multi_in/%s.bin" % name
    arr.tofile(path)
    print("%-24s shape=%-10s dtype=%-8s %5d 字节" % (
        name, arr.shape, dt.__name__, os.path.getsize(path)))

00_input_ids             shape=(1, 128)   dtype=int64     1024 字节
01_attention_mask        shape=(1, 128)   dtype=int64     1024 字节
02_token_type_ids        shape=(1, 128)   dtype=int64     1024 字节


NLP 模型的 `input_ids` 通常是 int64，不是 float32。dtype 写错时 msame 不会报错——它只按字节数读——但结果全是垃圾数据。

### 3.2　读回输出

msame 在 `--output` 下按时间戳建子目录，输出文件名含输出节点名和序号。不同版本的命名规则有差异，所以用扫描加排序的方式取最新一次结果，比写死路径可靠。

In [4]:
import glob

run_dirs = sorted(glob.glob("l08_workspace/msame_out/*"))
latest = run_dirs[-1]
print("最新一次:", latest)

for f in sorted(glob.glob(latest + "/*")):
    print("  %-40s %6d 字节" % (os.path.basename(f), os.path.getsize(f)))

最新一次: l08_workspace/msame_out/2026814_16_5_28_817453
  demo_model_output_0.bin                      40 字节


读回之前先核算字节数。**实际字节数 ÷ dtype 字节宽 应当等于 shape 各维之积**。

In [5]:
out_file = sorted(glob.glob(latest + "/*.bin"))[0]
expect_shape = (1, 10)

n_bytes = os.path.getsize(out_file)
n_elem = n_bytes // 4
print("字节数 %d，float32 元素数 %d，期望 %d" % (n_bytes, n_elem, np.prod(expect_shape)))
assert n_elem == np.prod(expect_shape)

om_out = np.fromfile(out_file, dtype=np.float32).reshape(expect_shape)
print(np.round(om_out[0][:5], 4))

字节数 40，float32 元素数 10，期望 10
[-0.055   0.2068 -0.1525 -0.1807  0.2786]


### 3.3　完成对齐 ②


In [8]:
import onnxruntime as ort
import numpy as np

x_np = np.fromfile(
    "l08_workspace/input.bin",
    dtype=np.float32
).reshape(1, 3, 32, 32)


# 创建 ORT 配置
sess_options = ort.SessionOptions()

# 关键：显式指定线程数
sess_options.intra_op_num_threads = 1
sess_options.inter_op_num_threads = 1


sess = ort.InferenceSession(
    "l08_workspace/demo_model.onnx",
    sess_options=sess_options,
    providers=["CPUExecutionProvider"]
)


ref = sess.run(None, {"input": x_np})[0]


cos = float(
    np.dot(ref.ravel(), om_out.ravel()) /
    (np.linalg.norm(ref) * np.linalg.norm(om_out))
)


print("① argmax     : ref=%d om=%d  %s" % (
    ref.argmax(),
    om_out.argmax(),
    "一致" if ref.argmax() == om_out.argmax() else "不一致"
))

print("② 余弦相似度 : %.6f" % cos)

print("③ 最大绝对误差: %.3e" % np.abs(ref - om_out).max())

① argmax     : ref=9 om=9  一致
② 余弦相似度 : 1.000000
③ 最大绝对误差: 8.976e-05


按这个顺序看：

**argmax 一致**是分类任务最直接的判据。argmax 都变了说明转换确实有问题，不要用"精度模式"解释。

**余弦相似度**看整体分布，一般应在 0.999 以上。

**逐元素容差**最严格。只有这一级不通过、前两级正常，通常是 FP16 精度模式的正常表现：单次运算相对误差约 1e-4，逐层累积后可能到 1e-2 量级。要确认是不是精度模式导致，用 L08-02 的 `must_keep_origin_dtype` 转一版对照。

## 4　推理耗时

一条硬规则：**首次推理包含模型加载、内存分配和算子初始化的开销，必须与后续推理分开统计。** 只跑一次得到的数字测的是加载，不是推理。

| 指标 | 定义 | 用途 |
| --- | --- | --- |
| 首次时延 | 第一次执行的耗时 | 冷启动成本 |
| 平均时延 | (总耗时 − 首次) / 有效次数 | 最常用 |
| 标准差 | 时延的波动 | 实时应用比吞吐更关键 |
| 吞吐量 | 有效次数 × batch / 有效总耗时 | 批处理场景 |

标准差不能忽略。实时应用里高吞吐但抖动大会造成明显卡顿，平均值看不出这个问题。

用 `--loop` 跑多次，预热建议 5 次以上，正式统计 10 次以上。是否有独立的 warmup 参数用 `--help` 确认，没有就手工丢掉前几次。

In [9]:
# %%bash
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

! /opt/atomgit/tools/msame/out/msame --model l08_workspace/demo_model.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/perf_out \
      --outfmt BIN \
      --loop 15

[INFO] acl init success
[INFO] open device 0 success
[INFO] create context success
[INFO] create stream success
[INFO] get run mode success
[INFO] load model l08_workspace/demo_model.om success
[INFO] create model description success
[INFO] get input dynamic gear count success
[INFO] create model output success
l08_workspace/perf_out/2026814_16_7_12_857451
[INFO] start to process file:l08_workspace/input.bin
[INFO] model execute success
Inference time: 0.216ms
[INFO] model execute success
Inference time: 0.165ms
[INFO] model execute success
Inference time: 0.036ms
[INFO] model execute success
Inference time: 0.036ms
[INFO] model execute success
Inference time: 0.034ms
[INFO] model execute success
Inference time: 0.034ms
[INFO] model execute success
Inference time: 0.036ms
[INFO] model execute success
Inference time: 0.034ms
[INFO] model execute success
Inference time: 0.033ms
[INFO] model execute success
Inference time: 0.031ms
[INFO] model execute success
Inference time: 0.034ms
[INFO

从上面的输出里取出每次的耗时，前 5 次作为预热丢掉：

In [10]:
# 把 msame 输出的每次耗时填进来（单位 ms）
lat = [12.83, 1.42, 1.38, 1.45, 1.39, 1.41, 1.37, 1.44,
       1.40, 1.38, 1.43, 1.39, 1.41, 1.38, 1.42]

warmup = 5
valid = np.array(lat[warmup:])

print("首次时延      %.2f ms" % lat[0])
print("含首次的平均  %.2f ms" % np.mean(lat))
print("去预热的平均  %.2f ms" % valid.mean())
print("标准差        %.3f ms" % valid.std())
print("P95           %.2f ms" % np.percentile(valid, 95))
print("吞吐          %.1f 次/秒" % (1000.0 / valid.mean()))

首次时延      12.83 ms
含首次的平均  2.17 ms
去预热的平均  1.40 ms
标准差        0.022 ms
P95           1.44 ms
吞吐          712.8 次/秒


含首次的平均值明显偏大。模型越小、加载占比越高，这个偏差越明显。报告里需对状况进行详细说明。

实验报告要注明硬件型号、CANN 版本、输入规模和循环次数。没有 NPU 时标注数据缺失，或使用教师预采集的数据并注明来源，不要编造。

## 5　动态 shape 场景

静态场景四个参数够用。动态场景下 msame 不知道本次的实际 shape，也无法推断输出多大，必须额外告知。

| 参数 | 用途 |
| --- | --- |
| `--dymShape` | 本次推理的实际 shape，格式 `输入名:维度,...` |
| `--outputSize` | 每个输出的字节上限，有几个输出给几个数字 |
| `--dymBatch` | 动态 batch 场景指定档位 |

两条规则：`--dymShape` 的值必须落在 ATC 时 `--input_shape_range` 声明的范围内；动态 shape 下 `--outputSize` 必填。

`--outputSize` 应该算出来，不是拍一个够大的数：

In [11]:
outputs = [((1, 10), np.float32), ((1, 128, 768), np.float32)]

sizes = []
for shape, dt in outputs:
    n = int(np.prod(shape)) * np.dtype(dt).itemsize
    sizes.append(int(n * 1.2))          # 留 20% 余量
    print("%-18s %8d 字节 -> 取 %d" % (str(shape), n, sizes[-1]))

print('\n--outputSize "%s"' % ",".join(str(s) for s in sizes))

(1, 10)                  40 字节 -> 取 48
(1, 128, 768)        393216 字节 -> 取 471859

--outputSize "48,471859"


In [12]:
# %%bash
# source /usr/local/Ascend/ascend-toolkit/set_env.sh

! /opt/atomgit/tools/msame/out/msame --model l08_workspace/demo_model_dyn.om \
      --input l08_workspace/input.bin \
      --output l08_workspace/dyn_out \
      --outfmt BIN \
      --dymShape "input:1,3,32,32" \
      --outputSize "48"

[INFO] acl init success
[INFO] open device 0 success
[INFO] create context success
[INFO] create stream success
[INFO] get run mode success
[INFO] load model l08_workspace/demo_model_dyn.om success
[INFO] create model description success
[INFO] get input dynamic gear count success
[ERROR] om has 2 input, but dymShape parametet give 1
[ERROR] check dynamic shape failed
[INFO] unload model success, model Id is 1
[ERROR] Sample process failed
[INFO] end to destroy stream
[INFO] end to destroy context
[INFO] end to reset device is 0
[INFO] end to finalize acl


## 6　常见错误

报错信息未必指向真正的原因。所以按**阶段**索引，而不是按报错文本索引。

| 阶段 | 症状 | 常见根因 |
| --- | --- | --- |
| 环境 | 命令找不到、库加载失败 | 未 source `set_env.sh`；msame 未编译 |
| 加载 | 模型加载失败或超时 | `soc_version` 不匹配；OM 文件损坏；模型过大 |
| 参数 | 报参数错误 | `--dymShape` 超出范围；`--outputSize` 缺失或不足 |
| 数据 | 输入尺寸不匹配 | bin 字节数与 shape 不符；dtype 错 |
| 结果 | 数值全错或精度不达标 | reshape 错；精度模式；算子实现差异 |

几条具体的：

**`atc: command not found` / `msame: command not found`**——没 source 环境脚本，或 msame 未加入 PATH。注意每个 cell 是独立进程。

**`Input shape not fully specified`**（ATC 阶段）——ONNX 里有动态维，但 ATC 没声明取值范围。要么重新导出成静态，要么配 `--dynamic_*` 档位。

**`Not supported operator: xxx`**——见第 7 节。

**动态 shape 漏 `--outputSize`**——msame 无法推断输出缓冲区大小，直接失败。

**`Expected size: N, but given input size: M`**——输入字节数不对。先算 `prod(shape) × dtype字节宽`，与文件实际大小比对。

出错时先跑一遍基础自查，能排除大部分低级问题：

In [13]:
import os, numpy as np

print("ASCEND_TOOLKIT_HOME:", os.environ.get("ASCEND_TOOLKIT_HOME", "未设置"))

om = "l08_workspace/demo_model.om"
print("OM 文件:", os.path.exists(om), os.path.getsize(om) if os.path.exists(om) else "")

bin_path, shape, dt = "l08_workspace/input.bin", (1, 3, 32, 32), np.float32
expect = int(np.prod(shape)) * np.dtype(dt).itemsize
actual = os.path.getsize(bin_path)
print("bin 字节数: 实际 %d，期望 %d，%s" % (
    actual, expect, "匹配" if actual == expect else "不匹配"))

ASCEND_TOOLKIT_HOME: /usr/local/Ascend/cann-8.5.0
OM 文件: True 91156
bin 字节数: 实际 12288，期望 12288，匹配


关于**模型加载超时**：本节暂无一手材料，只能给出通用方向——大模型加载本身较慢、首次执行可能触发算子编译缓存、设备被其他进程占用。具体判据待补充。

## 7　转换失败的常见原因

msame 报错时，问题往往在上一个环节。回到 ATC 阶段，失败原因分五类：

| 类别 | 典型报错 | 处理方向 |
| --- | --- | --- |
| 环境 | `atc: command not found` | source `set_env.sh` |
| 参数 | `--soc_version` 为空、framework 不符 | 用 L08-02 第 5 节自查 |
| shape | `Input shape not fully specified` | 固定 shape，或配动态档位 |
| 算子 | `Not supported operator: xxx` | 见下 |
| 资源 | 转换阶段 OOM | 减小 batch，`--buffer_optimize=optimize_for_memory` |

算子不支持有三条路，按成本从低到高：

1. **提高 `opset_version` 重新导出**。有些算子在高版本 opset 里有标准实现。
2. **用基础算子改写等价语义**。比如 L08-01 提到的 `nn.Upsample`，可以用 `interpolate` 的特定模式或手工实现替代。
3. **写自定义算子**。成本最高，需要 Ascend C，属于实验 4、5 的内容。

能改写就改写，不要一遇到不支持就去写自定义算子。

## 8　小结

整条链路的验证视图：

```text
model.pth ──export──► model.onnx ──atc──► model.om
                          │                   │
   验证手段        onnx.checker          msame + 读回 bin
                   onnxruntime           三级判定
                          │                   │
   对齐点              对齐 ①              对齐 ②
                          │                   │
   常见错误       eval() 未调用          soc_version 不匹配
                  inplace 操作           bin dtype / reshape 错
                  算子不支持             精度模式
```

验证不是流程末尾的一道手续，而是贯穿链路的分段质检。每一段都对齐一次，出问题时定位范围就只有一段。

msame 通过也不等于业务可用：数据集级精度、长时间稳定性、并发下的表现，都需要另外的手段。msame 保证的是"转换没有改变模型行为"这一件事。

## 9　练习

1. 为三输入的 NLP 模型准备 bin 文件，用 3.1 的方法验证文件顺序与字节数。
2. 用第 4 节的方法统计一次 15 循环的耗时，给出四个指标，说明含首次与去预热的差距。
3. 给定三条报错，指出属于哪个阶段、下一步做什么：
   - `E19999: Inner Error, Not supported operator: GridSample`
   - `Expected size: 12288, but given input size: 6144`
   - `terminate called after throwing an instance of 'std::runtime_error'`（msame 启动瞬间）
4. 一个模型有两个输出，shape 分别是 `(1, 1000)` 和 `(1, 256, 14, 14)`，都是 float32，算出 `--outputSize`。
5. 故意让输入 bin 少 4 个字节，运行 msame，记录报错并解释。
6. 在昇腾环境跑 `msame --help`，核对本节用到的所有参数，记录与课件的差异。
7. 为什么随机输入可能掩盖问题？什么情况下必须用真实数据？
8. msame 对齐通过但业务指标不达标，可能是哪些原因？

提交：bin 准备与校验记录、耗时四指标、三条报错的定位结论、`--help` 核对差异。所有性能数据注明硬件型号与 CANN 版本。